In [1]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


# 1. Environment
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing in .env")

In [2]:
# 2. Financial data
financial_data = {
    "ТехНова Інк": (
        "ТехНова Інк за 2024 рік: "
        "виручка 450 млн дол (+18 відсотків), "
        "чистий прибуток 67 млн дол, "
        "маржа EBITDA 22 відсотки. "
        "Працівників 2300 осіб. "
        "ROE становить 15.2 відсотка. "
        "Співвідношення борг до EBITDA дорівнює 1.8. "
        "Штаб-квартира у Львові."
    ),
    "ГрінЕнерджі Солюшнс": (
        "ГрінЕнерджі Солюшнс за 2024 рік: "
        "виручка 890 млн дол (+35 відсотків), "
        "чистий прибуток 156 млн дол, "
        "маржа EBITDA 28 відсотків. "
        "Працівників 5100 осіб. "
        "ROE становить 19.7 відсотка. "
        "Співвідношення борг до EBITDA дорівнює 0.9. "
        "Лідер у секторі відновлюваної енергії."
    ),
    "РітейлМакс Груп": (
        "РітейлМакс Груп за 2024 рік: "
        "виручка 1200 млн дол (+8 відсотків), "
        "чистий прибуток 48 млн дол, "
        "маржа EBITDA 12 відсотків. "
        "Працівників 8500 осіб. "
        "ROE становить 9.3 відсотка. "
        "Співвідношення борг до EBITDA дорівнює 3.2. "
        "Мережа з 340 магазинів в Україні."
    ),
    "ФінТех Дайнамікс": (
        "ФінТех Дайнамікс за 2024 рік: "
        "виручка 230 млн дол (+52 відсотки), "
        "чистий збиток мінус 15 млн дол, "
        "маржа EBITDA мінус 8 відсотків. "
        "Працівників 890 осіб. "
        "Від'ємний показник ROE. "
        "Стартап, який активно інвестує "
        "у масштабування бізнесу."
    ),
}


In [3]:
# 3. Create documents with metadata
documents = [
    Document(
        page_content=content,
        metadata={
            "company": company,
            "year": 2024,
            "category": "фінансові звіти компаній",
        },
    )
    for company, content in financial_data.items()
]

In [4]:
# 4. Split into logical chunks
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=2000,
    chunk_overlap=0,
)

chunks = text_splitter.split_documents(documents)

assert len(chunks) == 4, (
    f"Expected 4 chunks, got {len(chunks)}"
)

print(f"Created {len(chunks)} chunks")

for index, chunk in enumerate(chunks, start=1):
    print(f"Chunk {index}: {chunk.metadata['company']}")



Created 4 chunks
Chunk 1: ТехНова Інк
Chunk 2: ГрінЕнерджі Солюшнс
Chunk 3: РітейлМакс Груп
Chunk 4: ФінТех Дайнамікс


In [5]:
# 5. Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [6]:
# 6. Chroma vector store
vector_store = Chroma(
    collection_name="finansovi_zvity_kompaniy",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={
        "description": "Фінансові звіти компаній за 2024 рік"
    },
)

# Stable IDs prevent duplicates on repeated runs
vector_store.add_documents(
    documents=chunks,
    ids=[
        "company_technova_2024",
        "company_greenenergy_2024",
        "company_retailmax_2024",
        "company_fintech_2024",
    ],
)

print(f"Documents in Chroma: {vector_store._collection.count()}")

assert vector_store._collection.count() == 4


Documents in Chroma: 4


In [7]:
# 8. LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

In [8]:
# 9. RAG prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Ти фінансовий аналітик.

Відповідай виключно українською мовою.

Правила:
1. Використовуй лише наданий контекст.
2. Не вигадуй фінансових показників.
3. Зберігай точні числа та одиниці вимірювання.
4. Не плутай показники різних компаній.
5. Якщо інформація відсутня, відповідай:
   "На основі наданого контексту,
   інформація відсутня."
6. Відповідай коротко та по суті.

Контекст:
{context}
""",
        ),
        ("human", "{question}"),
    ]
)


In [ ]:
# 10. Generation chain
generation_chain = prompt | llm | StrOutputParser()


def format_documents(docs):
    return "\n\n".join(
        f"Компанія: {doc.metadata['company']}\n"
        f"{doc.page_content}"
        for doc in docs
    )

In [20]:
# 11. RAG function

# Create retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

def rag_chain(question: str) -> str:
    # Retrieve 2 most relevant documents
    retrieved_docs = retriever.invoke(question)

    # Identify the company requested by the user
    requested_company = next(
        (
            company
            for company in financial_data
            if company.casefold() in question.casefold()
        ),
        None,
    )

    # Protect against answers about unknown companies
    if requested_company is None:
        return (
            "На основі наданого контексту, "
            "інформація про запитувану "
            "компанію відсутня."
        )

    # Check that the company was actually retrieved
    matching_docs = [
        doc
        for doc in retrieved_docs
        if doc.metadata["company"] == requested_company
    ]

    if not matching_docs:
        return (
            "На основі знайденого контексту, "
            "інформація про запитувану "
            "компанію відсутня."
        )

    context = format_documents(matching_docs)

    return generation_chain.invoke(
        {
            "context": context,
            "question": question,
        }
    )


In [17]:
# 12. Test questions
test_cases = [
    (
        "Яка виручка у компанії ТехНова Інк за 2024 рік?",
        "450",
    ),
    (
        "Скільки працівників у ГрінЕнерджі Солюшнс?",
        "5100",
    ),
    (
        "Який показник ROE у РітейлМакс Груп?",
        "9.3",
    ),
    (
        "Де знаходиться штаб-квартира ТехНова Інк?",
        "Львів",
    ),
    (
        "Яка маржа EBITDA у ФінТех Дайнамікс?",
        "8",
    ),
    (
        "Який чистий прибуток у ЕкоПакування Солюшнс?",
        "відсутня",
    ),
]


In [21]:
# 13. Run and evaluate
print("\n" + "=" * 70)

passed = 0

for index, (question, expected) in enumerate(
    test_cases, start=1
):
    answer = rag_chain(question)

    # Accept both "Львів" and "Львові"
    if expected == "Львів":
        is_correct = (
            "Львів" in answer or "Львові" in answer
        )
    else:
        is_correct = expected.lower() in answer.lower()

    passed += int(is_correct)

    print(f"\nПитання {index}: {question}")
    print(f"Відповідь: {answer}")
    print(f"Тест: {'PASS' if is_correct else 'FAIL'}")
    print("-" * 70)

print(f"\nРезультат: {passed}/{len(test_cases)}")




Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.
Тест: PASS
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
Тест: PASS
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
Тест: PASS
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
Тест: PASS
----------------------------------------------------------------------

Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс за 2024 рік становить мінус 8 відсотків.
Тест: PASS
-------------------------------------------------------------------